# Equilibrium shortcut vs. brute force — *same answer, and how?*

Two questions, answered head-on:

1. **Do the fast shortcut (`lunar/equilibrium.py`) and a long brute-force
   spin-up give the same result?** → Demonstration 1 overlays the profiles.
2. **How does the shortcut reach the true steady state without the long
   integration?** → explained just below, then demonstrated.

## How the shortcut reaches the true value

The true periodic steady state is pinned by **two physical facts** — not by
waiting:

1. **The surface skin** repeats a daily cycle set by sunlight, and it settles
   *fast* (~10 lunations).
2. **Deep down**, at steady state there are no heat sources in the rock, so the
   *same* heat flows through every layer: the geothermal flux $Q_b$ from below.
   That **fixes the temperature gradient at every depth**:
   $$\frac{d\langle T\rangle}{dz} = \frac{Q_b}{K(\langle T\rangle,\,z)}.$$

**Key insight:** the deep profile's *shape* is not a free unknown you must
discover by waiting — it is fixed by energy conservation.

- **Brute force** discovers it slowly: start from an arbitrary temperature and
  let heat physically diffuse for ~$10^3$ lunations until the deep gradient
  relaxes to $Q_b/K$. The waiting is just the rock forgetting its arbitrary
  start (diffusion through 5 m is slow).
- **The shortcut** *uses* the gradient law instead of waiting for it:
  1. a **short** spin-up (~12 lunations) settles the fast skin and gives the
     correct temperature at one **anchor** depth (~0.55 m);
  2. from that anchor it **integrates $dT/dz = Q_b/K$ straight down** to fill
     the entire deep profile *instantly*;
  3. feed that back into another short spin-up and repeat — 3–5 rounds, because
     the surface temperature barely depends on the deep column.

The only thing it "assumes" is the gradient law — and that law is **exact** at
steady state (energy conservation + the bottom boundary condition). So both
methods *must* land on the same profile.

**Analogy:** filling a tank to find its steady level. Brute force pours water in
and waits for the sloshing to stop; the shortcut knows the level is set by
inflow = outflow and computes it directly. Same level — one just doesn't wait.

In [ ]:
import sys, os, time, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts" / "figures"))
import make_equilibrium_demo as demo
print("project root:", ROOT, "| CPU cores:", os.cpu_count())

## Demonstration 1 — same answer (the overlay)

Run brute force toward convergence from **two different starting temperatures**
(240 K and 260 K) and overlay the final profiles on the shortcut. If the
mechanism above is right, all three curves lie on top of each other.

`N_LUN` sets how far brute force runs. The committed reference figure
[`output/figures/fig_equilibrium_demo.pdf`](../output/figures/fig_equilibrium_demo.pdf)
was made at **N_LUN = 3000** (agreement **60–95 mK** over the sensor depths —
the small residual is brute force still crawling the last bit; the shortcut
reached the same place in ~9 s). Increase `N_LUN` to shrink the residual;
decrease it for a quicker look. Runtime ≈ `N_LUN`×0.09 s per run, two runs in
parallel.

In [ ]:
N_LUN        = 2000          # lunations per brute-force run (3000 ~ 60-95 mK; bump higher to tighten)
GUESSES      = (240.0, 260.0)
USE_PARALLEL = True          # two runs across two cores; set False to run serially

In [ ]:
t0 = time.perf_counter()
res = demo.converged_profiles(n_lun=N_LUN, guesses=GUESSES, parallel=USE_PARALLEL)
print(f"shortcut: {res['shortcut']['wall']:.1f} s   |   brute force: {time.perf_counter()-t0:.0f} s wall\n")
for g in sorted(res["profiles"]):
    print(f"  start {g:.0f} K  ->  agrees with shortcut to "
          f"{res['resid_sensor'][g]*1e3:5.1f} mK over sensor depths "
          f"(0.8-2.4 m),  {res['resid_full'][g]*1e3:4.0f} mK over the full column")

In [ ]:
fig = demo.plot_overlay(res)
fig

**How to read it.** In panel (a) the thick grey shortcut profile and both dashed
brute-force profiles (from 240 K and 260 K) are indistinguishable — two very
different starts and the direct method all reach one profile. Panel (b) shows
the leftover difference: tens of mK, and it is purely brute force *still
converging* (the deep middle of the column relaxes slowest). The shortcut hits
this in ~9 s; brute force needs thousands of lunations to match it.

## Demonstration 2 (optional) — *why* brute force is slow

The convergence *rate*: how the brute-force error shrinks as lunations are
added, from each starting guess. This is the "waiting for diffusion" cost the
shortcut avoids. (Independent runs across `N_GRID` × `GUESSES` are fanned over
all your cores.)

In [ ]:
res2 = demo.compute_curves(n_grid=(50, 100, 200, 400, 800),
                           guesses=GUESSES, parallel=USE_PARALLEL)
fig2 = demo.plot_curves(res2)
fig2

## Takeaway

- **Same destination:** brute force from any start converges onto the shortcut's
  profile — they solve the same steady-state problem.
- **~30× less work:** the shortcut reaches it in ~9 s; brute force needs
  ~3000 lunations (~4 min) to match it to the shortcut's certified tolerance.
- **And it's certified:** the shortcut returns `drift` and `flux_closure`
  diagnostics proving it arrived — you don't have to guess how many lunations
  is "enough" (the trap that produced the original biased result, flag F1).